# DataService Tutorial — get_data Module

This notebook documents the **`get_data`** function from the **data_service** module in MultiNEAs. Use it to fetch asteroid/comet and fireball data from two JPL sources:

1. **CNEOS** — [NASA JPL Fireball API](https://ssd-api.jpl.nasa.gov/doc/fireball.html): bright fireballs/bolides detected by sensors.
2. **SBDB** — [NASA JPL SBDB Query API](https://ssd-api.jpl.nasa.gov/doc/sbdb_query.html): Small-Body Database (asteroids, comets, NEOs, PHAs).

Call **`get_data(source, params)`** with the desired source and API parameters; it returns a **DataTable** (pandas DataFrame plus metadata).

## 1. Import and basic usage

- **get_data(source, params)** — A single function that fetches data from the given `source` (`'CNEOS'` or `'SBDB'`) using the `params` dict. Returns a **DataTable** with:
  - **`.data`** — pandas DataFrame of the results
  - **`.source`** — `"CNEOS"` or `"SBDB"`
  - **`.metadata`** — the query params used

In [1]:
from multineas.data_service import get_data

Welcome to MultiNEAs v0.3.5


---
## 2. CNEOS (Fireball API)

**Source:** `source='CNEOS'`  
**API:** [JPL Fireball API](https://ssd-api.jpl.nasa.gov/doc/fireball.html) — fireball/bolide events (date, energy, impact energy, location, altitude, velocity).

### 2.1 CNEOS query parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| **date-min** | string | Exclude data *earlier than* this (`YYYY-MM-DD` or `YYYY-MM-DDThh:mm:ss`) |
| **date-max** | string | Exclude data *later than* this date |
| **energy-min** | string | Exclude radiated energy *less than* this (in 10¹⁰ J) |
| **energy-max** | string | Exclude radiated energy *greater than* this |
| **impact-e-min** | string | Exclude impact energy *less than* this (kilotons, kt) |
| **impact-e-max** | string | Exclude impact energy *greater than* this (kt) |
| **alt-min** | number | Exclude altitude *less than* this (km) |
| **alt-max** | number | Exclude altitude *greater than* this (km) |
| **req-loc** | boolean | If true, only records with latitude/longitude |
| **req-alt** | boolean | If true, only records with altitude |
| **req-vel-comp** | boolean | If true, only records with velocity components (vx, vy, vz) |
| **vel-comp** | boolean | If true, include vx, vy, vz in the response |
| **sort** | string | Sort by: `date`, `energy`, `impact-e`, `vel`, `alt`; prefix `-` for descending (default `-date`) |
| **limit** | number | Return only the first N results |

In [2]:
# Example: fireballs with impact energy between 0.1 and 0.5 kt, limit 5
source = 'CNEOS'
params = {
    "impact-e-min": "0.1",
    "impact-e-max": "0.5",
    "limit": 5
}
cneos_data = get_data(source, params)
cneos_data.data

,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel
0,2026-01-31 18:07:14,5.8,0.19,4.1,N,173.4,W,32.0,NaN
1,2026-01-30 10:25:37,3.4,0.12,45.0,S,174.5,E,89.0,71.1
2,2025-12-16 20:58:12,9.6,0.29,24.1,S,92.4,W,25.0,19.0
3,2025-11-15 00:48:43,10.5,0.32,62.2,S,94.7,W,30.0,16.0
4,2025-11-11 17:39:51,9.3,0.28,27.3,N,79.8,W,42.0,18.7


### 2.2 More CNEOS examples (date range, req-loc/req-alt)

Filter by date and require location and altitude for mapping.

In [3]:
# Fireballs in a date range, with location and altitude (good for mapping)
params_date = {
    "date-min": "2025-01-01",
    "date-max": "2025-02-01",
    "req-loc": True,
    "req-alt": True,
    "limit": 10
}
cneos_date = get_data('CNEOS', params_date)
print("Records:", len(cneos_date.data))
cneos_date.data

Records: 2


,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel
0,2025-01-11 14:46:00,2.9,0.1,1.2,S,55.3,E,61.0,13.2
1,2025-01-10 21:11:07,2.7,0.095,47.2,N,106.3,E,34.6,14.0


---
## 3. SBDB (Small-Body Database Query API)

**Source:** `source='SBDB'`  
**API:** [JPL SBDB Query API](https://ssd-api.jpl.nasa.gov/doc/sbdb_query.html) — asteroids, comets, NEOs, PHAs with orbital elements and other fields.

### 3.1 SBDB query parameters

| Parameter | Description |
|-----------|-------------|
| **fields** | Comma-separated list of output fields (required for data; if omitted only count is returned). Examples: `full_name`, `epoch`, `a`, `e`, `i`, `om`, `w`, `ma`, `q`, `Q`, `n`, `tp`, `per`, `M` |
| **sb-kind** | `a` = asteroids only, `c` = comets only |
| **sb-group** | `neo` = Near-Earth Objects, `pha` = Potentially Hazardous Asteroids |
| **sb-class** | Orbit class: `IEO`, `ATE`, `APO`, `AMO`, `MBA`, `CEN`, `TNO`, etc. (comma-separated for multiple) |
| **sb-ns** | `n` = numbered only, `u` = unnumbered only |
| **sort** | Sort by up to 3 fields; prefix `-` for descending (e.g. `-a`) |
| **limit** | Return only the first N records |
| **limit-from** | Start from record N (pagination); requires `limit` |
| **full-prec** | If set, use full-precision numeric output |

In [4]:
# Example: NEO asteroids — orbital elements (a, e, i, om, w, ma, q)
source = 'SBDB'
params = {
    "fields": "full_name,epoch,a,e,i,om,w,ma,q",
    "sb-kind": "a",     # a = asteroids, c = comets
    "sb-group": "neo",  # neo = NEOs, pha = Potentially Hazardous Asteroids
    "limit": 5
}
sbdb_data = get_data(source, params)
sbdb_data.data

,full_name,epoch,a,e,i,om,w,ma,q
0,433 Eros (A898 PA),2461000.5,1.458,0.2228,10.83,304.27,178.93,310.55,1.133
1,719 Albert (A911 TB),2461000.5,2.637,0.5466,11.57,183.86,156.19,240.61,1.195
2,887 Alinda (A918 AA),2461000.5,2.474,0.5712,9.40,110.41,350.53,81.54,1.061
3,1036 Ganymed (A924 UB),2461000.5,2.665,0.5332,26.68,215.44,132.50,97.59,1.244
4,1221 Amor (1932 EA1),2461000.5,1.92,0.4346,11.87,171.24,26.76,59.87,1.085


### 3.2 More SBDB examples (PHAs, sort, sb-class)

Potentially Hazardous Asteroids with selected fields; or NEOs sorted by semi-major axis.

In [5]:
# PHAs (Potentially Hazardous Asteroids) — name, a, e, q, limit 5
params_pha = {
    "fields": "full_name,a,e,q",
    "sb-kind": "a",
    "sb-group": "pha",
    "limit": 5
}
sbdb_pha = get_data('SBDB', params_pha)
sbdb_pha.data

,full_name,a,e,q
0,1566 Icarus (1949 MA),1.078,0.8270,0.186
1,1620 Geographos (1951 RA),1.246,0.3355,0.828
2,1862 Apollo (1932 HA),1.471,0.5599,0.647
3,1981 Midas (1973 EA),1.776,0.6505,0.621
4,2101 Adonis (1936 CA),1.874,0.7641,0.442


In [6]:
# NEOs sorted by semi-major axis descending (largest first), limit 5
params_sort = {
    "fields": "full_name,a,e,i",
    "sb-kind": "a",
    "sb-group": "neo",
    "sort": "-a",
    "limit": 5
}
sbdb_sort = get_data('SBDB', params_sort)
sbdb_sort.data

,full_name,a,e,i
0,(2017 UR52),350.3,0.9964,108.26
1,(A/2024 G8),144.3,0.9919,97.41
2,(2016 XK24),132.5,0.9904,145.63
3,(2019 EJ3),96.58,0.9888,139.98
4,(A/2019 Q2),59.68,0.9789,159.03


---
## 4. Working with the DataTable result

`get_data` always returns a **DataTable** with:

- **`.data`** — pandas DataFrame (use `.head()`, `.columns`, plot, export, etc.).
- **`.source`** — `"CNEOS"` or `"SBDB"`.
- **`.metadata`** — the `params` dict you passed (useful for reproducibility).

In [7]:
# Inspect the result of a previous call (e.g. CNEOS)
result = cneos_data
print("Source:", result.source)
print("Params used:", result.metadata)
print("Shape:", result.data.shape)
result.data.head()

Source: CNEOS
Params used: {'impact-e-min': '0.1', 'impact-e-max': '0.5', 'limit': 5}
Shape: (5, 9)


,date,energy,impact-e,lat,lat-dir,lon,lon-dir,alt,vel
0,2026-01-31 18:07:14,5.8,0.19,4.1,N,173.4,W,32.0,NaN
1,2026-01-30 10:25:37,3.4,0.12,45.0,S,174.5,E,89.0,71.1
2,2025-12-16 20:58:12,9.6,0.29,24.1,S,92.4,W,25.0,19.0
3,2025-11-15 00:48:43,10.5,0.32,62.2,S,94.7,W,30.0,16.0
4,2025-11-11 17:39:51,9.3,0.28,27.3,N,79.8,W,42.0,18.7


---
## 5. Invalid input parameters

Each source accepts only a fixed set of parameter **names**. If you pass any key that is not in that set, `get_data` raises a **ValueError** with the invalid keys and the list of valid parameters.

- **CNEOS** valid names: `date-min`, `date-max`, `energy-min`, `energy-max`, `impact-e-min`, `impact-e-max`, `alt-min`, `alt-max`, `req-loc`, `req-alt`, `req-vel-comp`, `vel-comp`, `sort`, `limit`.
- **SBDB** valid names: `fields`, `sb-kind`, `sb-group`, `sb-class`, `sb-ns`, `sort`, `limit`, `limit-from`, `full-prec`.

Examples below show invalid parameters and the resulting error.

In [8]:
# CNEOS: using an SBDB-only parameter (e.g. "sb-group") raises ValueError
try:
    get_data("CNEOS", {"sb-group": "neo", "limit": 5})
except ValueError as e:
    print("ValueError (expected):", e)

ValueError (expected): Invalid parameter(s): ['sb-group']. Valid parameters for CNEOS: ['alt-max', 'alt-min', 'date-max', 'date-min', 'energy-max', 'energy-min', 'impact-e-max', 'impact-e-min', 'limit', 'req-alt', 'req-loc', 'req-vel-comp', 'sort', 'vel-comp']


In [9]:
# SBDB: using a CNEOS-only parameter (e.g. "impact-e-min") raises ValueError
try:
    get_data("SBDB", {"fields": "full_name,a,e", "impact-e-min": "0.1", "limit": 5})
except ValueError as e:
    print("ValueError (expected):", e)

ValueError (expected): Invalid parameter(s): ['impact-e-min']. Valid parameters for SBDB: ['fields', 'full-prec', 'limit', 'limit-from', 'sb-class', 'sb-group', 'sb-kind', 'sb-ns', 'sort']


---
## 6. Summary and references

- **get_data(source, params)** — use `source='CNEOS'` or `source='SBDB'` and pass the correct **params** for each API.
- **CNEOS** params: `date-min`, `date-max`, `energy-min/max`, `impact-e-min/max`, `alt-min/max`, `req-loc`, `req-alt`, `req-vel-comp`, `vel-comp`, `sort`, `limit`.
- **SBDB** params: `fields` (required for data), `sb-kind`, `sb-group`, `sb-class`, `sb-ns`, `sort`, `limit`, `limit-from`, `full-prec`.
- Result is a **DataTable**: `.data` (DataFrame), `.source`, `.metadata`.

**API docs:**  
- [Fireball API (CNEOS)](https://ssd-api.jpl.nasa.gov/doc/fireball.html)  
- [SBDB Query API](https://ssd-api.jpl.nasa.gov/doc/sbdb_query.html)